# Demo — Predict only (use a shipped cardiac classifier)

This is the recommended first-touch notebook for TissueTypist. It runs
**prediction only**, using one of the shipped pre-trained cardiac
presets. No training step.

The shipped `default` preset was trained on Visium SD (3-prime + FFPE)
and Visium HD reference data; you can apply it to any Visium SD query
without retraining.

If your query is from an imaging-based platform (Xenium / MERFISH /
CosMx) where the gene panel is much smaller than the reference, you
need to **retrain on the panel** instead — see `demo_merfish.ipynb`.


## 1. Setup

In [ ]:
# Edit this path to point at your query Visium AnnData (raw counts in adata.X).
from pathlib import Path

QUERY     = Path.home() / "anndata" / "my_visium.h5ad"   # ← change me
OUTDIR    = Path("results/predict_only_demo")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Optional: pick a preset other than "default" — alternatives are
# "own_only" (no neighbour features) and "neighbour_heavy".
PRESET    = "default"


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import tissuetypist
from tissuetypist import predict_adata, load_preset

print("TissueTypist version:", tissuetypist.__version__)
print("Preset:", PRESET, "→", load_preset(PRESET))


## 2. Load and inspect the query

Required:

- **Raw counts in `adata.X`** (no log-normalisation — TissueTypist handles
  this automatically based on dtype heuristics).
- **`obsm['spatial']`** — 2-D coordinates per spot.
- **`obs[section_col]`** — a section / library identifier; spots from
  different sections must not share coordinates.


In [ ]:
adata = sc.read_h5ad(QUERY)
print(adata)
print()
print("var_names:", adata.var_names[:5].tolist(), "...")
print("obs columns:", adata.obs.columns.tolist())


In [ ]:
# Pick the section column. Check `adata.obs.columns` if `section_ID`
# isn't the right one for your dataset.
SECTION_COL = "section_ID"

n_spots_per_section = adata.obs[SECTION_COL].value_counts()
print(n_spots_per_section)


## 3. Run prediction

In [ ]:
adata = predict_adata(
    adata,
    model_dir=load_preset(PRESET),
    modality="sd",                 # "sd" for Visium SD; "hd" for Visium HD windows
    section_col=SECTION_COL,
)

# All output is written into adata.obs as `tt_*` columns.
tt_cols = [c for c in adata.obs.columns if c.startswith("tt_")]
print(f"{len(tt_cols)} tt_* columns added: {tt_cols}")


## 4. Inspect predictions

`tt_final_label` is the recommended per-spot label (the finest class
that resolved confidently). `tt_coarse_score` is the confidence of the
coarse-level prediction.


In [ ]:
# Top labels by spot count
print(adata.obs["tt_final_label"].value_counts().head(15))


In [ ]:
# Save the predictions for downstream use
out_h5ad = OUTDIR / "predicted.h5ad"
adata.write_h5ad(out_h5ad)
print(f"Saved → {out_h5ad}")

# CSV summary of just the obs predictions
out_csv = OUTDIR / "predictions.csv.gz"
cols = [SECTION_COL] + tt_cols
adata.obs[cols].to_csv(out_csv, index=True)
print(f"Saved → {out_csv}")


## 5. Spatial plot

In [ ]:
sections = sorted(adata.obs[SECTION_COL].unique())
fig, axes = plt.subplots(1, len(sections),
                         figsize=(4 * len(sections), 4),
                         squeeze=False)

for ax, sec in zip(axes[0], sections):
    sub = adata[adata.obs[SECTION_COL] == sec]
    coords = sub.obsm["spatial"]
    labels = sub.obs["tt_final_label"].astype("category")

    # Stable colour mapping
    cats = labels.cat.categories
    palette = dict(zip(cats, plt.get_cmap("tab20").colors[: len(cats)]))
    colors = labels.map(palette).tolist()

    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=2, linewidths=0)
    ax.set_aspect("equal"); ax.set_title(sec, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

# Single legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=palette[k], label=k) for k in cats]
fig.legend(handles=handles, loc="center right",
           bbox_to_anchor=(1.18, 0.5), fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(OUTDIR / "spatial_predictions.pdf", bbox_inches="tight")
plt.show()


## 6. Confidence distribution

In [ ]:
# Coarse-level confidence per spot. Spots with low coarse_score
# fall back to the coarse label; spots with high coarse_score resolve
# to a fine class.
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(adata.obs["tt_coarse_score"], bins=40, color="steelblue", edgecolor="white")
ax.axvline(0.5, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("tt_coarse_score")
ax.set_ylabel("spots")
ax.set_title("Coarse-level confidence")
fig.savefig(OUTDIR / "coarse_score_hist.pdf", bbox_inches="tight")
plt.show()


## Where to next

- **Output column reference** — every `tt_*` column in
  [`docs/output-columns.md`](../docs/output-columns.md).
- **Niche hierarchy** — what each label means and how the multi-stage
  sub-models work, in [`docs/hierarchy.md`](../docs/hierarchy.md).
- **Imaging-based ST** — for Xenium / MERFISH / CosMx queries where
  retraining is required, see [`demo_merfish.ipynb`](demo_merfish.ipynb).
- **Evaluation** — to compute confusion matrices against ground truth
  labels, use `tissuetypist evaluate` (see
  [`docs/user-guide.md`](../docs/user-guide.md)).
